# TRIDENT T69 â€” dense 4Ã— baseline CNN, two-epoch pilot
Preparation only. Training requires explicit approval and is disabled by default. No evaluation results are included.

GitHub/source archive â†’ Colab code; Drive dataset ZIP â†’ local `/content` cache; completed epoch checkpoints â†’ Drive; CPU inference export â†’ Windows FastAPI.

Select **Runtime â†’ Change runtime type â†’ GPU**. See `docs/colab_training.md` for Windows packaging, dependency choices, resume limitations and approval steps. Do not install `requirements-learning-cpu.txt` here.

## 1. Settings and runtime
Fill the exact reviewed Git commit (after publishing the local files), or use the source archive alternative. Keep RUN_ID unchanged when reconnecting. Leave TRAINING_APPROVED=False until the first run is explicitly approved.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, zipfile
SOURCE_MODE = "github"  # or "archive" for reviewed, unpublished local source
GIT_URL = "https://github.com/Dhanush1756/TRIDENT-Depth-T69.git"
GIT_COMMIT = ""  # exact full 40-character reviewed commit; do not use a moving branch
SOURCE_SHA256 = ""  # required only for source archive mode
DATASET_SHA256 = ""  # printed by scripts/pack_colab.py on Windows
DRIVE_ROOT = Path("/content/drive/MyDrive/TRIDENT-T69")
REPO = Path("/content/TRIDENT-Depth-T69")
CACHE = Path("/content/trident-nyu-converted")
RUN_ID = "pilot_dense4_lightweight_seed69"
RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID
RESUME = ""  # e.g. "epoch_001.pt"; latest complete checkpoint only
TRAINING_APPROVED = False
EXPORT_APPROVED = False
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use a Python 3.10â€“3.13 runtime"
print(platform.python_version(), platform.platform())


## 2. Mount persistent Drive
Upload `nyu_converted.zip` and optionally `trident_source.zip` under DRIVE_ROOT. Archive roots must contain the converted files / repository files directly, without an extra folder. Never store credentials in the notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert DRIVE_ROOT.is_dir(), f"Create {DRIVE_ROOT} and upload the archives first"


## 3. Obtain reviewed source
The current local work is not yet on GitHub. Publish the reviewed changes yourself before using GitHub mode, or package them with the documented source command. Existing checkout directories are not overwritten.

In [ ]:
if SOURCE_MODE == "github":
    assert len(GIT_COMMIT) == 40 and all(c in "0123456789abcdefABCDEF" for c in GIT_COMMIT), "Set the reviewed full Git commit"
    if not REPO.exists():
        subprocess.run(["git", "clone", GIT_URL, str(REPO)], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", GIT_COMMIT], check=True)
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    assert actual.lower() == GIT_COMMIT.lower()
    assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Checkout has local modifications"
    SOURCE_ID = actual
elif SOURCE_MODE == "archive":
    archive = DRIVE_ROOT / "trident_source.zip"
    assert len(SOURCE_SHA256) == 64, "Set source archive SHA256"
    digest = hashlib.sha256()
    with archive.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    assert digest.hexdigest() == SOURCE_SHA256
    assert not REPO.exists(), "Use a fresh runtime or GitHub mode for an existing checkout"
    with zipfile.ZipFile(archive) as bundle:
        names = set()
        for item in bundle.infolist():
            name = item.filename
            target = (REPO / name).resolve()
            assert "\\" not in name and target.is_relative_to(REPO.resolve())
            assert name not in names and (item.external_attr >> 16) & 0o170000 != 0o120000
            names.add(name)
        bundle.extractall(REPO)
    SOURCE_ID = SOURCE_SHA256
else:
    raise ValueError("SOURCE_MODE must be github or archive")
assert (REPO / "trident_depth/training/colab.py").is_file(), "Reviewed Colab changes are missing"
os.chdir(REPO)


Use the existing Colab Python runtime directly, preserving its CUDA-enabled PyTorch. Only additional requirements are installed; any dependency plan touching PyTorch or its CUDA stack is rejected. The repository installs with dependency resolution disabled. Fresh subprocesses use the updated packages without creating a virtual environment. Record exact Python, torch, CUDA, package versions and GPU, and run the complete project test suite after installation. Resume requires the same software fingerprint; hardware changes may still affect numerical reproducibility.

In [ ]:
# Use Colab's existing runtime directly: no venv or ensurepip.
import tempfile
PY = sys.executable
print("Colab Python:", PY)
torch_probe = """
import json, torch
print(json.dumps(dict(version=str(torch.__version__), cuda=torch.version.cuda,
 location=torch.__file__, cuda_available=torch.cuda.is_available())))
"""
TORCH_BEFORE = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_BEFORE["cuda_available"], "Select a GPU runtime; Colab CUDA PyTorch must already be available"

# Resolve only the additional requirements first. Reject any plan that would
# touch PyTorch or its CUDA stack, then install exactly that plan without resolving
# further dependencies. This cannot pull in a CPU-only torch wheel.
with tempfile.TemporaryDirectory(prefix="trident-pip-") as temporary:
    report = Path(temporary) / "install-plan.json"
    subprocess.run([str(PY), "-m", "pip", "install", "--dry-run",
                    "--only-binary=:all:", "--report", str(report),
                    "-r", str(REPO / "requirements-colab.txt")], check=True)
    plan = json.loads(report.read_text())["install"]
    protected = {"torch", "torchvision", "torchaudio", "torchtext", "triton", "pytorch-triton"}
    for item in plan:
        name = item["metadata"]["name"].lower().replace("_", "-")
        assert name not in protected and not name.startswith("nvidia-"), f"Refusing to replace Colab GPU package: {name}"
    urls = [item["download_info"]["url"] for item in plan]
    if urls:
        subprocess.run([str(PY), "-m", "pip", "install", "--no-deps",
                        "--only-binary=:all:", *urls], check=True)
subprocess.run([str(PY), "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
TORCH_AFTER = json.loads(subprocess.check_output([str(PY), "-c", torch_probe], text=True))
assert TORCH_AFTER == TORCH_BEFORE, "Colab PyTorch/CUDA changed during setup"
probe = """
import json, platform, torch, numpy, scipy, cv2, PIL, h5py
assert (2, 6) <= tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (3, 0)
print(json.dumps(dict(python=platform.python_version(), torch=str(torch.__version__),
 cuda=torch.version.cuda, numpy=numpy.__version__, scipy=scipy.__version__,
 opencv=cv2.__version__, pillow=PIL.__version__, h5py=h5py.__version__,
 gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)))
"""
ENVIRONMENT = json.loads(subprocess.check_output([str(PY), "-c", probe], text=True))
print(json.dumps(ENVIRONMENT, indent=2))
assert ENVIRONMENT["gpu"], "No CUDA GPU available: select a GPU runtime. Do not run the pilot on CPU."
# Complete project suite after installation; CPU tests do not train a model.
subprocess.run([str(PY), "-m", "pytest", "-q"], check=True,
               env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))


## 5. Stage and audit the official dataset
Copy/extract from the uploaded ZIP into temporary Colab storage once per runtime. Archive SHA256 and the approved manifest SHA256 are checked. Preserve 672 training / 123 validation / 654 locked test records and their scene assignments. Test pixels are not loaded by training or validation.

In [ ]:
assert len(DATASET_SHA256) == 64, "Set the dataset archive SHA256 printed on Windows"
stage = """
import sys, json
from pathlib import Path
from trident_depth.training.colab import stage_archive, audit_pilot_dataset
archive, cache, expected = sys.argv[1:]
if not Path(cache).exists():
    stage_archive(archive, cache, expected)
assert (Path(cache) / '.archive_sha256').read_text() == expected, 'Cache belongs to another archive'
print(json.dumps(audit_pilot_dataset(Path(cache) / 'manifest_development.json'), indent=2))
"""
subprocess.run([str(PY), "-c", stage, str(DRIVE_ROOT / "nyu_converted.zip"),
                str(CACHE), DATASET_SHA256], check=True)
MANIFEST = CACHE / "manifest_development.json"


## 6. Inspect unchanged pilot and provenance
No optimizer steps. Actual pilot uses full 480Ã—640 frames, FP32, Adam, batch 1, seed 69, learning rate 1e-4, two epochs, validation on all 123 development validation frames, and best global valid-pixel RMSE in metres. The constant scheduler keeps that learning rate unchanged.

In [ ]:
EXPERIMENT = REPO / "configs/phase5/pilot_dense4_lightweight.json"
cfg = json.loads(EXPERIMENT.read_text())
pipeline_file = EXPERIMENT.parent / cfg["pipeline_config"]
pipeline = json.loads(pipeline_file.read_text())
assert (cfg["model"], cfg["epochs"], cfg["batch_size"], cfg["learning_rate"], cfg["seed"]) == ("lightweight_cnn", 2, 1, 1e-4, 69)
assert (pipeline["protocol"], pipeline["scale"]) == ("dense_sr", 4)
print(json.dumps({"experiment": cfg, "pipeline": pipeline}, indent=2))
RECORD = {"source_id": SOURCE_ID, "environment": ENVIRONMENT,
          "dataset_archive_sha256": DATASET_SHA256,
          "experiment_sha256": hashlib.sha256(EXPERIMENT.read_bytes()).hexdigest(),
          "pipeline_sha256": hashlib.sha256(pipeline_file.read_bytes()).hexdigest()}
ENV_FILE = RUN_DIR.parent / (RUN_ID + "_environment.json")
if ENV_FILE.exists():
    assert json.loads(ENV_FILE.read_text()) == RECORD, "Resume provenance differs; restore original source/environment/data"
if RESUME:
    assert (RUN_DIR / RESUME).is_file(), "Selected complete checkpoint is missing"
else:
    assert not RUN_DIR.exists(), "Existing run: set RESUME to the latest complete epoch checkpoint"


## 7. Guarded launch or resume â€” disabled
After explicit approval, set TRAINING_APPROVED=True in section 1 and rerun the cells. Checkpoints go directly to Drive. On reconnect: rerun setup with identical source, dataset, environment and RUN_ID, then set RESUME to the latest complete `epoch_XXX.pt`. Ignore `.part` files. A fully completed two-epoch pilot has no remaining epochs; proceed to export. Interrupted epochs are repeated, not resumed mid-batch.

In [ ]:
if not TRAINING_APPROVED:
    print("Training disabled. No optimizer steps performed.")
else:
    assert ENVIRONMENT["gpu"] and MANIFEST.is_file()
    RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not ENV_FILE.exists():
        ENV_FILE.write_text(json.dumps(RECORD, indent=2))
        freeze = subprocess.check_output([str(PY), "-m", "pip", "freeze"], text=True)
        ENV_FILE.with_suffix(".freeze.txt").write_text(freeze)
    command = [str(PY), "-m", "trident_depth.training.train",
               "--experiment", str(EXPERIMENT), "--manifest", str(MANIFEST),
               "--output", str(RUN_DIR), "--device", "auto", "--execute-training"]
    if RESUME:
        command += ["--resume", str(RUN_DIR / RESUME)]
    subprocess.run(command, check=True)


## 8. Export existing trained weights for Windows â€” disabled
Export only after reviewing the pilot. Full Drive checkpoints retain model, Adam, constant scheduler, CPU/CUDA/Python/NumPy/DataLoader RNG and completed history. The separate CPU inference export omits optimizer/RNG and loads through the existing FastAPI checkpoint loader. It cannot resume training. No fabricated weights or metrics are generated.

In [ ]:
if not EXPORT_APPROVED:
    print("Export disabled; no checkpoint created.")
else:
    selection = "from trident_depth.training.colab import best_checkpoint_path; import sys; print(best_checkpoint_path(sys.argv[1]))"
    source = Path(subprocess.check_output([str(PY), "-c", selection, str(RUN_DIR)], text=True).strip())
    destination = RUN_DIR / "baseline_dense4_inference.pt"
    subprocess.run([str(PY), "-m", "trident_depth.training.colab",
                    str(source), str(destination)], check=True)
    print("Download to Windows:", destination)
